# Methods

Summer4 stand-in for `tweak_cases_calib` `notebooks/06-Documentation.ipynb`.

The original notebook rendered Google-style `Notes:` sections through
`document.get_func_notes`. Those Notes were not ported onto every summer4
builder. This notebook states the same topics against the modules that exist
here, and prints the fixed-parameter and prior tables from `parameters.py`.

Default model: corrected replacement deaths, progression weights that sum to
one, `rel_infectiousness_subclin` on subclinical disease, folded spline
contact, smooth entry. `match_golden=True` restores the three summer3 bugs
plus year-frozen mixing and step entry for golden parity. Solver backends live
in `vacf.solvers` (`Diffrax` + `PIDController`); calibration uses
`CALIBRATION_SOLVER`.

Use the **vacf (pixi)** kernel.


## Demographics

Age breaks are `AGE_STRATA` in `vacf.constants`. UN population, mortality
and fertility loaders are in `vacf.inputs`; entry rates and smooth vs step
birth lookups are in `vacf.demography`. Replacement (non-TB) deaths feed
age 0 on the default path.

## Natural history and detection

Flows are declared in `vacf.model.build_model`: infection via
`ForceOfInfection`, latency and clinical transitions, cosine detection
scale-up (`vacf.curves.get_cos_multicurve`), treatment and ACF hazards
filled through `derived_fn`, triangular seeding, and ageing.

## Mixing

Static year-frozen construction is `vacf.mixing.get_norm_c_matrix`. The
default run folds a cubic spline (`vacf.contact` + `fold_spline_contact` /
`spline_mixing` in `model.py`) once per parameter draw.

## Calibration

`vacf.calibration.make_log_likelihood` is the seven-term summer3 density
(latent, notifications, TB deaths, adult pulmonary prevalence, survey decline,
high-infectious share, Canberra mixing distance). It is hand-rolled: summer4
`BayesianModel` stock likelihoods do not yet express those transforms. Targets
are in `vacf.targets`.


## Fixed parameters


In [ ]:
import pandas as pd

from vacf.parameters import BASE_PARAMS, PARAM_BOUNDS, PARAM_NAMES

fixed = {k: v for k, v in BASE_PARAMS.items() if k not in PARAM_BOUNDS}
pd.DataFrame(
    {
        "parameter": [PARAM_NAMES.get(k, k) for k in fixed],
        "name": list(fixed),
        "value": list(fixed.values()),
    }
)


## Prior ranges


In [ ]:
pd.DataFrame(
    {
        "parameter": [PARAM_NAMES.get(k, k) for k in PARAM_BOUNDS],
        "name": list(PARAM_BOUNDS),
        "low": [PARAM_BOUNDS[k][0] for k in PARAM_BOUNDS],
        "high": [PARAM_BOUNDS[k][1] for k in PARAM_BOUNDS],
        "base": [BASE_PARAMS[k] for k in PARAM_BOUNDS],
    }
)
